In [1]:
# Imports
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import duckdb
from config.settings import DUCKDB_FILE
from src.database.database_manager import DatabaseManager

# Initialize database manager
db = DatabaseManager()

## Data Quality: Salary Outliers

`SALARY_MAX_THRESHOLD` (previously capping `salary_max` at 100,000 SGD) was removed from the cleaning pipeline so unusually high salaries are no longer silently dropped. That surfaces a real data-quality issue in `SGJobData.csv`: a small number of postings have `salary_max` values that are obviously not real monthly SGD salaries (e.g. in the millions), most likely data-entry or scraping errors.

The cells below quantify how many rows are affected and how much they distort `AVG(salary_max)`.

In [2]:
# Overall salary distribution stats
stats = db.query("""
    SELECT
        COUNT(*) as total_rows,
        MIN(salary_max) as min_salary_max,
        MAX(salary_max) as max_salary_max,
        AVG(salary_max) as avg_salary_max,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary_max) as median_salary_max,
        STDDEV(salary_max) as stddev_salary_max
    FROM jobs
""")
stats

,total_rows,min_salary_max,max_salary_max,avg_salary_max,median_salary_max,stddev_salary_max
0,548799,1000,25330000,6257.151463,4500.0,69257.938976


In [3]:
# How many rows look like data errors, at a few thresholds?
threshold_counts = db.query("""
    SELECT
        COUNT(*) FILTER (WHERE salary_max > 20000) as over_20k,
        COUNT(*) FILTER (WHERE salary_max > 50000) as over_50k,
        COUNT(*) FILTER (WHERE salary_max > 100000) as over_100k,
        COUNT(*) FILTER (WHERE salary_max > 1000000) as over_1m
    FROM jobs
""")
threshold_counts

,over_20k,over_50k,over_100k,over_1m
0,6469,385,249,12


In [4]:
# The worst offenders, to eyeball what the "error" actually looks like
top_outliers = db.query("""
    SELECT job_id, title, company, salary_min, salary_max
    FROM jobs
    ORDER BY salary_max DESC
    LIMIT 20
""")
top_outliers

,job_id,title,company,salary_min,salary_max
0,job_8fcfd8ac0914,Accounts Executive - GT,RK RECRUITMENT PTE. LTD.,2800,25330000
1,job_1d3028808edf,Executive Secretary,SAFRAN LANDING SYSTEMS SERVICES SINGAPORE PTE....,14719,23712119
2,job_df437bc040d4,Language Teacher,ASCOTT INTERNATIONAL MANAGEMENT PTE LTD,324072,20862169
3,job_6b111aa4d2ec,"Sales Associate (Home Audio, Retail)",RK RECRUITMENT PTE. LTD.,262482,15531134
4,job_cb7310ff666c,sales and operations manager,THALES DIS (SINGAPORE) PTE. LTD.,164428,14420727
5,job_aa0c76629e2f,Social media content creator,MINDFLEX EDUCATION PTE. LTD.,260117,13798518
6,job_67efc9b62e0f,Resident Physician,ASCEND INTERNATIONAL TRAINING PTE. LTD.,108872,10734314
7,job_73a657db01d3,Clinic assistant,HILL GROVE MEDICAL PTE. LTD.,1500,10000000
8,job_9eb2a8056c7d,Junior Project Manager (IT Infrastructure),RECRUIT EXPRESS PTE LTD,267303,7859259
9,job_85f9475917d5,Senior Manager - Operations,BOND CAPITAL GROUP PTE. LTD.,107908,6142101


In [5]:
# Quantify the distortion: AVG(salary_max) with vs. without the >100k outliers
impact = db.query("""
    SELECT
        (SELECT AVG(salary_max) FROM jobs) as avg_with_outliers,
        (SELECT AVG(salary_max) FROM jobs WHERE salary_max <= 100000) as avg_without_outliers,
        (SELECT COUNT(*) FROM jobs WHERE salary_max > 100000) as outlier_rows,
        (SELECT COUNT(*) FROM jobs) as total_rows
""")
impact['pct_rows_are_outliers'] = (impact['outlier_rows'] / impact['total_rows'] * 100).round(3)
impact['avg_inflated_by'] = (impact['avg_with_outliers'] - impact['avg_without_outliers']).round(2)
impact

,avg_with_outliers,avg_without_outliers,outlier_rows,total_rows,pct_rows_are_outliers,avg_inflated_by
0,6257.151463,5900.703715,249,548799,0.045,356.45
